# W16 Exercise: Build an evaluation set that exposes a bug

Machine Learning in Engineering Science | NCKU / LAiMM | Fall 2026

**Date:** 2026-12-21. **Exercise time:** 35 minutes within the class/practice allocation, not extra time after class.

**Classroom formative practice. No new weekly graded submission is created.**

## How to work independently
1. In Colab, save a copy to your Drive and use a CPU runtime. In Jupyter, open this notebook and select a Python kernel.
2. Run the complete worked walkthrough below from top to bottom. Read every Markdown checkpoint; the walkthrough is included so no previous notebook state is needed.
3. At the Independent Exercise section, write each prediction in the response cell before running its experiment.
4. Run the starter experiment, make the requested controlled change, and record the actual result. Edit only the declared factor; preserve split rules and preprocessing.
5. Explain one mechanism and one limitation. Export the evidence JSON and discuss it with a partner.

The core exercises need no paid API, private dataset, or GPU. The agent units use clearly labeled offline simulators, not real LLM calls.

**Reproducibility boundary:** the original guided notebooks were previously executed; the new extension cells have not yet been execution-tested in this revision. Report unexpected behavior rather than claiming a check passed.


# Week 16: Agent Studio II: an evaluation harness

Machine Learning in Engineering Science | NCKU + LAiMM | Fall 2026

**Purpose:** a fully worked, self-paced in-class practice with an optional extension. Weeks 11 and 18 are optional examination-readiness materials. These notebooks do not create weekly graded assignments.

**Before running:** choose a Python 3 kernel in Jupyter or a CPU runtime in Colab. Run every cell from top to bottom with Shift+Enter. Setup installs missing free packages. No API key or external dataset download is required.

**Learning target:** complete the calculation, inspect intermediate values, and explain why the protocol supports a limited claim.

**Time:** about 25-45 minutes including reading, prediction, execution, and explain-back.


## 1. Set up and record the environment

Keep the seed fixed for the first run. Record actual outputs and package versions rather than copying a claimed performance number.


In [ ]:
import sys, subprocess, importlib.util, platform, json, random, hashlib
from pathlib import Path
required = {"numpy": "numpy", "matplotlib": "matplotlib", "sklearn": "scikit-learn"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
import numpy as np
import matplotlib.pyplot as plt
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
print("Python:", platform.python_version(), "NumPy:", np.__version__, "Seed:", SEED)


**Checkpoint.** The environment cell must finish before any analysis cell. Restart and run all if a variable is missing.


## 2. Construct reference cases before evaluation

Use fixed arithmetic tasks as an inexpensive workflow test. This does not evaluate natural-language understanding.


In [ ]:
cases=[{"id":"a","values":[1,2,3],"expected":2.0},
       {"id":"b","values":[-2,2],"expected":0.0},
       {"id":"c","values":[4,4,4],"expected":4.0}]
def baseline(values): return float(np.mean(values))
def buggy_candidate(values): return float(sum(values)/max(len(values)-1,1))


**Checkpoint.** Expected answers are fixed independently of the candidate implementation.


## 3. Measure correctness with a stated tolerance

An evaluation harness runs identical inputs and compares against the same ground truth. Record per-case failures, not only a summary.


In [ ]:
rows=[]
for case in cases:
    for name,fn in [("baseline",baseline),("candidate",buggy_candidate)]:
        observed=fn(case["values"])
        rows.append({"id":case["id"],"system":name,"observed":observed,"expected":case["expected"],
                     "pass":bool(np.isclose(observed,case["expected"],atol=1e-8))})
for row in rows: print(row)


**Checkpoint.** The intentionally buggy candidate should fail nonzero-mean cases. A denominator error has a concrete mechanism.


## 4. Create an explainable failure report

Do not promote a candidate just because it looks plausible or passes one zero-mean case.


In [ ]:
summary={name:{"passed":sum(r["pass"] for r in rows if r["system"]==name),"total":len(cases)}
         for name in ["baseline","candidate"]}
summary["decision"]="Keep baseline; fix candidate denominator before retesting."
print(json.dumps(summary,indent=2))


**Checkpoint.** Describe why the zero-mean case alone would miss the bug. Add edge cases before expanding the workflow.


## 6. Preserve your evidence

Export a JSON record with a seed, package version, and measured result. Download the artifacts directory before leaving Colab; runtime storage is temporary.


In [ ]:
record={"week":16,"seed":SEED,"numpy":np.__version__,"python":platform.python_version(),"result":summary}
(ARTIFACTS/"week-16-evidence.json").write_text(json.dumps(record,indent=2))
print(json.dumps(record,indent=2))


**Checkpoint.** The evidence file must match the result you actually obtained. A saved screenshot alone does not permit rerunning the calculation.


## 7. Try one change and explain it
1. Predict the effect of one change before editing code. Change one parameter, input, or boundary condition only.
2. Rerun all cells in a fresh kernel. Keep the original evidence record and compare the observed difference.
3. Explain the result using the mechanism, not just the final number.
4. Name one assumption that limits the claim, and a new test that could challenge it.

**Troubleshooting:** if installation fails, restore network access; if a variable is undefined, run earlier cells; if your output differs, compare package versions, seed, and data construction. Do not repeatedly edit code until a preferred score appears.

**Completion criterion:** one executed notebook, one evidence JSON, and a brief claim/evidence/limitation statement.

### Reference documentation
- [NumPy user guide](https://numpy.org/doc/stable/user/)
- [scikit-learn user guide](https://scikit-learn.org/stable/user_guide.html)
- [Course repository](https://github.com/CHjonnyYu/MLES-Fall-2026)


## Independent Exercise: W16

Complete the three tasks below after the walkthrough. The starter cells produce observations, not your written interpretation. Fill the response fields in the next cell. Save separate records for the original run and any controlled change.

In [ ]:
exercise_id="W16"
observations={}
responses={
    'task1_prediction':'', 'task1_explanation':'',
    'task2_prediction':'', 'task2_explanation':'',
    'task3_prediction':'', 'task3_explanation':'',
    'controlled_change':'', 'limitation':'', 'ai_use_and_checks':''
}
# Fill these strings before and after each experiment. Use 'none' if no AI assistance was used.
print('Exercise record initialized:', exercise_id)

### Task 1: Calculate before comparing candidates

Compute the mean of [1,2,3] by hand. Compare the reference and intentionally buggy candidate.

**Steps**
1. Write your prediction in `responses['task1_prediction']` before running.
2. Run the starter cell; read every printed quantity and retain the observation.
3. Make the controlled comparison requested above. Record its configuration in `responses['controlled_change']`.
4. Write a mechanism-based explanation in `responses['task1_explanation']`; do not merely repeat a score.

**Checkpoint:** The expected mean is 2. The candidate must be judged against the reference, not against its own output.

In [ ]:
case_values=[1,2,3]
observations['task1']={'expected':2.,'baseline':float(baseline(case_values)),'candidate':float(buggy_candidate(case_values))}
print(observations['task1'])

### Task 2: Expose a misleading passing case

Compare zero-sum and negative-mean cases. Explain why passing one symmetric case can hide a denominator bug.

**Steps**
1. Write your prediction in `responses['task2_prediction']` before running.
2. Run the starter cell; read every printed quantity and retain the observation.
3. Make the controlled comparison requested above. Record its configuration in `responses['controlled_change']`.
4. Write a mechanism-based explanation in `responses['task2_explanation']`; do not merely repeat a score.

**Checkpoint:** The zero-sum case can pass despite the bug. At least one nonzero-mean case is needed to expose the failure.

In [ ]:
observations['task2']=[]
for values_ex in [[-2,2],[-2,-4],[5]]:
    observations['task2'].append({'values':values_ex,'expected':float(np.mean(values_ex)),'candidate':float(buggy_candidate(values_ex))})
print(observations['task2'])

### Task 3: Declare a numerical tolerance

Compare two close values under two tolerances, and explain the difference between absolute and relative error.

**Steps**
1. Write your prediction in `responses['task3_prediction']` before running.
2. Run the starter cell; read every printed quantity and retain the observation.
3. Make the controlled comparison requested above. Record its configuration in `responses['controlled_change']`.
4. Write a mechanism-based explanation in `responses['task3_explanation']`; do not merely repeat a score.

**Checkpoint:** The comparison passes at absolute tolerance 1e-6 but fails at 1e-9. A tolerance must be justified before interpreting pass rates.

In [ ]:
expected_ex=1.; observed_ex=1.+1e-7
observations['task3']={str(tol):bool(np.isclose(observed_ex,expected_ex,atol=tol,rtol=0)) for tol in [1e-6,1e-9]}
print(observations['task3'])

### Finish: preserve and explain

Fill all response strings, then run the export cell. A draft file is intentionally labeled incomplete; a complete response record is not automatically a scientifically correct result. Download the JSON before the Colab session ends. Share it with a partner for explain-back, not as an additional weekly submission.

**Exit questions:** What changed? Which observation supports your explanation? What does this experiment not establish?

In [ ]:
missing_responses=[key for key,value in responses.items() if not str(value).strip()]
exercise_record={'exercise':exercise_id,'status':'draft' if missing_responses else 'ready for peer discussion','responses':responses,'observations':observations,'missing_responses':missing_responses,'environment':{'python':platform.python_version(),'numpy':np.__version__},'data_scope':'course-provided or synthetic teaching data'}
exercise_folder=Path('artifacts')/'exercises'; exercise_folder.mkdir(parents=True,exist_ok=True)
exercise_path=exercise_folder/(exercise_id+'-evidence.json')
exercise_path.write_text(json.dumps(exercise_record,indent=2,default=lambda value:value.tolist() if hasattr(value,'tolist') else str(value)))
print('Saved:',exercise_path)
print('Status:',exercise_record['status'])
print('Fields still to complete:',missing_responses)